In [3]:
import os
from cerebras.cloud.sdk import Cerebras
import pandas as pd
import numpy as np
import json
from config_gen import generate_coalitional_conf, generate_divergent_conf, generate_minorty_conf, generate_uniform_conf
from strats import ADD, MPL, APP, LMS, MAJ, FAI,MAJ_from_df, BORDA, AWM, BORDA_from_df
import random
import time
import re
from ollama import Client
from cerebras.cloud.sdk import RateLimitError


In [ ]:
client_ol = Client() #requires ollama logged in on device (mac). Pipeline, however, is adaptable for e.g, Huggingface. Only the LLM call needs to be switched.


### Helper functions which are needed further down


def dcg_at_k(relevance_scores, k=10):
    relevance_scores = np.array(relevance_scores)[:k]
    return np.sum(relevance_scores / np.log2(np.arange(2, len(relevance_scores) + 2)))

def ndcg_at_k(predicted_order, gold_order, k=10, binary_relevance=False):

    if binary_relevance:
        gold_set = set(gold_order)
        predicted_relevance = [1 if item in gold_set else 0 for item in predicted_order]
        ideal_relevance = [1] * len(gold_order)  
    else:
        relevance_map = {item: len(gold_order) - i for i, item in enumerate(gold_order)}
        predicted_relevance = [relevance_map.get(item, 0) for item in predicted_order]
        ideal_relevance = [relevance_map[item] for item in gold_order]
    
    dcg = dcg_at_k(predicted_relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg



In [ ]:
## Initialization of group generation (configurations) and social choice-based aggregation strategies

random.seed(time.time())

num_items = 50
group_size = 4
domains = ['tourist locations', 'movies', 'anon']

configuration_list = ['divergent', 'uniform', 'coalitional', 'minority']
configurations = {
    "coalitional": lambda: generate_coalitional_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "divergent":   lambda: generate_divergent_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "minority":    lambda: generate_minorty_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "uniform":     lambda: generate_uniform_conf(n=group_size, m=num_items, r=100, options=ITEMS),
}

strategies = {
    "ADD": lambda df: ADD(df),
    "MAJ": lambda df: MAJ_from_df(df),
    "LMS": lambda df: LMS(df),
    "MPL": lambda df: MPL(df),
    "APP": lambda df: APP(df, threshold=60),
    #"FAI": lambda: FAI(result),
    "BORDA": lambda df: BORDA_from_df(df),
    "AWM": lambda df: AWM(df, threshold=35),
}

Datasets: 

Low risk: Movielens 
High risk: Tourist destinations

In [ ]:

preprocessed_dataset_folder = "datasets/movielens_dataset"
threshold = 0.75 ## popularity threshold (percentile) to only include most popular movie titles/tourist destinations

m_ratings = pd.read_csv(preprocessed_dataset_folder + "/ratings.csv")
m_titles = pd.read_csv(preprocessed_dataset_folder + "/movies.csv")

rating_counts = (
    m_ratings
    .groupby('item')
    .size()
    .rename('rating_count')
)

rating_cutoff = rating_counts.quantile(threshold) ## In the current study, we only use the most popular movies.

selected_items = rating_counts[
    rating_counts >= rating_cutoff
].index.tolist()

selected_ratings_df = m_ratings[m_ratings['item'].isin(selected_items)]
m_included = m_titles[m_titles['item'].isin(selected_items)]

titles = m_included['title'].tolist()

print(f"{len(m_included)} movie titles included | threshold n ratings: {rating_cutoff:.0f}")

tourist = pd.read_csv('datasets/Tourist_Destinations.csv')
tourist.columns = [
    "Destination Name","Country","Continent","Type",
    "Avg Cost (USD/day)","Best Season","Avg Rating",
    "visits","UNESCO Site"
]

visit_cutoff = tourist['visits'].quantile(threshold)

tourist = tourist[tourist['visits'] >= visit_cutoff]

tourist['Loc'] = tourist['Destination Name'] + ', ' + tourist['Country']
locations = tourist['Loc'].tolist()

print(f"{len(locations)} locations included | threshold visits: {visit_cutoff:.2f}")


1192 movie titles included | threshold n ratings: 14
504 locations included | threshold visits: 7.51


In [ ]:
## Test run for generating a group and link domain names to the synthetic group matrix

configuration = random.choice(configuration_list)
domain = random.choice(domains)
print(configuration,':' ,domain)

if domain == 'movies':
    ITEMS = random.sample(titles, num_items)
elif domain == 'anon':
    ITEMS = [f"item_{i}" for i in range(1, num_items + 1)]
else:
    ITEMS = random.sample(locations, num_items)
group_matrix = configurations[configuration]()


divergent : movies


In [ ]:
###########################
### SCENARIO GENERATION ###
###########################

# This cell is only used to generate a new group dataset. Repository includes the original groups (groups.csv)
# running this is unnecessary as groups are already included. But code can be used to generate additional data


goal = 50
ids = 0
file_path = 'datasets/groups_2.csv' ## adjusted name to _2 so you do not overwrite original file

if os.path.exists(file_path):
    df_results = pd.read_csv(file_path)
else:
    columns = ['configuration', 'num_items','group']
    df_results = pd.DataFrame(columns=columns)

item_set_sizes = [25,50,75]

for conf in configuration_list:
    print(f'#### STARTING {conf} ####')
    
    for item_count in item_set_sizes:
        print(f'######## STARTING {item_count} item set size ##########')
        count = 0
        while count < goal:
            ITEMS = [f"item_{i}" for i in range(1, item_count + 1)]
            num_items = item_count
            matrix = configurations[conf]()
            restaurants = matrix[0].keys()
            result = {r: [entry[r] for entry in matrix] for r in restaurants}
            #print(result)
            
            df_results = pd.concat([df_results, pd.DataFrame([{
        'groupID': int(ids),
        'configuration': conf,
        'num_items':num_items,
        'group': str(result),
        }])], ignore_index=True)
            df_results.to_csv('groups.csv', index=False)
            count +=1
            ids +=1

        

#### STARTING divergent ####
######## STARTING 25 item set size ##########
######## STARTING 50 item set size ##########
######## STARTING 75 item set size ##########
#### STARTING uniform ####
######## STARTING 25 item set size ##########
######## STARTING 50 item set size ##########
######## STARTING 75 item set size ##########
#### STARTING coalitional ####
######## STARTING 25 item set size ##########
######## STARTING 50 item set size ##########
######## STARTING 75 item set size ##########
#### STARTING minority ####
######## STARTING 25 item set size ##########
######## STARTING 50 item set size ##########
######## STARTING 75 item set size ##########


In [ ]:
##########################
####### MAIN LOOP ########
##########################

# First, checks whether the results dataset already exists. If so, loop carries on where you left off previously
# Main loop formats the group and feeds it to the specified LLM
# output is formatted in json and added to results.csv


import ast
#goal = 10000
count = 0

file_path = 'results.csv'
completed = set()

if os.path.exists(file_path):
    df_results = pd.read_csv(file_path)

    completed = set(
        zip(
            df_results['groupID'],
            df_results['domain'],
            df_results['llm']
        )
    )
else:
    columns = ['groupID','configuration', 'group_size','num_items','domain',
               'llm', 'recommendation', 'explanation',
               'ADD', 'APP', 'LMS', 'MPL', 'MAJ', 'group']
    df_results = pd.DataFrame(columns=columns)

dfs = pd.read_csv('groups.csv')
llms = ["ministral-3:8b-cloud",'mistral-large-3:675b-cloud', "gpt-oss:20b-cloud", "gpt-oss:120b-cloud"]
## error catchers ##
errorcount = 0
ers = []
###############

def parse_llm_json(text: str) -> dict:
    try:
        return json.loads(text)

    except json.JSONDecodeError:
        cleaned = re.sub(r"```(?:json)?", "", text).strip()

        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in LLM output")

        return json.loads(match.group())

for row in dfs.itertuples():
    try:
        configuration = row.configuration
        num_items = row.num_items
        group = ast.literal_eval(row.group)
        groupid = row.groupID
        for domain in ['anon', 'movies', 'tourist']:
            if domain == 'movies':
                ITEMS = random.sample(titles, num_items)
            elif domain == 'anon':
                ITEMS = [f"item_{i}" for i in range(1, num_items + 1)]
            else:
                ITEMS = random.sample(locations, num_items)
            for llm_name in llms:
                if (groupid, domain, llm_name) in completed:
                    continue
                result = dict(zip(ITEMS, group.values()))        
                rating_rows = []
                for item, ratings in result.items():
                    for r in ratings:
                        if isinstance(r, list):
                            rating_rows.extend([{"item": item, "rating": v} for v in r])
                        else:
                            rating_rows.append({"item": item, "rating": r})

                rating_df = pd.DataFrame(rating_rows)
                strategy_outputs = {}
                strategy_outputs = {s: strategies[s](rating_df) for s in ['ADD','APP','LMS','MPL','MAJ']}
                system_message = {
                                'role':'system',
                                'content': f"""
            You are tasked with making group recommendations based on the different preferences of the group members. 
            You need explain the process behind making the recommendation to the group in such a way that someone without recommender systems knowledge can understand. 
            The information you are provided contain the {domain} preferences of the group. Every candidate item for recommendation has a rating from each user listed in the order by user (first rating from user1, second from user2 etc). 
            The rating is a scale from 0 to 100. For the recommendation, you simply mention the {domain}.
            You make a recommendation to the group of users by providing a ranking of 10 {domain} based on the recommendation approach you came up with. 

            Provide your answer as VALID JSON ONLY.
            Do not use markdown or code fences.
            Do not include newlines inside string values.
            Use plain ASCII characters only.

            Format:
                {{
                "recommendation": ["item1","item2","item3","item4","item5","item6","item7","item8","item9","item10"],
                "explanation": Short explanation of how you made the recommendation with no line breaks"
                }}

            """}
                messages = []
                messages.append(system_message)
                scenario = {
                    'role': 'user',
                    'content': f"""
                    The per-item ratings are presente below:
                    ### BEGIN TABLE ###
                    {result}
                    ### END TABLE ###

                    Think about the answer internally, but only output the final JSON object (containing recommendation ranking and explanation). Do not include any additional text or python code. 
                    Return STRICT JSON. Do not use markdown.
                    """

                }
                messages.append(scenario)
                chat_instance = messages
                if 'gpt' in llm_name.lower(): #gpt oss
                    resp = client_ol.chat(llm_name, messages=messages,options={"temperature":0.5})
                    out = resp['message']['content']
                elif 'stral' in llm_name.lower(): #mistral, ministral
                    resp = client_ol.chat(llm_name, messages=messages,options={"temperature":0.5})
                    out = resp['message']['content']    
                else:
                    resp = client.chat.completions.create(
                                                messages = chat_instance,
                                                model=llm_name,
                                                stream=False,
                                                max_completion_tokens=1000,
                                                temperature=0.5,
                                            )

                    out = resp.choices[0].message.content
                if "<think>" in out or "</think>" in out:
                    out = re.sub(r'^.*?</think>', '', out, flags=re.DOTALL).strip()
                out = parse_llm_json(out)
                #print(out)
                recommendation = out['recommendation']
                explanation = out['explanation']

                row = {
                'configuration': configuration,
                'domain': domain,
                'llm': llm_name,
                'recommendation': recommendation,
                'explanation': explanation
                    }

                for strategy in ['ADD', 'APP', 'LMS', 'MPL', 'MAJ']:
                    row[strategy] = strategy_outputs[strategy]

                new_row = pd.DataFrame([{
                    'groupID': groupid,
                    'configuration': configuration,
                    'group_size': group_size,
                    'num_items': num_items,
                    'domain': domain,
                    'llm': llm_name,
                    'recommendation': recommendation,
                    'explanation': explanation,
                    'ADD': row['ADD'],
                    'APP': row['APP'],
                    'LMS': row['LMS'],
                    'MPL': row['MPL'],
                    'MAJ': row['MAJ'],
                    'group': str(result),
                }])

                if df_results.empty:
                    df_results = new_row
                else:
                    df_results = pd.concat([df_results, new_row], ignore_index=True)

                df_results.to_csv(file_path, index=False)

                time.sleep(2)
    except RateLimitError as e:
        errorcount +=1
        msg = str(e)
        ers.append(msg)
        if errorcount > 5:
            print('Time to stop. Error count above 5')
            print(ers)
            break
        
        if "high traffic" in msg.lower() or "queue_exceeded" in msg.lower() or "limit" in msg.lower():
            print(f"High traffic detected.")
            time.sleep(20)
            continue



In [94]:
import ollama as ol
from importlib.metadata import version
version('rapidfuzz')

'3.12.2'

In [ ]:
if os.path.exists(file_path):
    df_results = pd.read_csv(file_path)
else:
    columns = ['configuration', 'group_size','num_items','domain','llm', 'recommendation', 'explanation', 'ADD', 'APP', 'LMS', 'MPL', 'MAJ', 'group']
    df_results = pd.DataFrame(columns=columns)

dfs = pd.read_csv('groups.csv')

def parse_llm_json(text: str) -> dict:
    try:
        return json.loads(text)

    except json.JSONDecodeError:
        cleaned = re.sub(r"```(?:json)?", "", text).strip()

        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in LLM output")

        return json.loads(match.group())


## error catchers ##
errorcount = 0
ers = []
###############

while count <goal:
    try: 
        count+=1
        configuration = random.choice(configuration_list)
        domain = random.choice(domains)
        group_size = 4
        num_items = random.choice([25,50,75])
        print(configuration,'-', group_size, '-',num_items,':' ,domain)

        if domain == 'movies':
            ITEMS = random.sample(titles, num_items)
        elif domain == 'anon':
            ITEMS = [f"item_{i}" for i in range(1, num_items + 1)]
        else:
            ITEMS = random.sample(locations, num_items)

        matrix = configurations[configuration]()
        system_message = {
                        'role':'system',
                        'content': f"""
    You are tasked with making group recommendations based on the different preferences of the group members. 
    You need explain the process behind making the recommendation to the group in such a way that someone without recommender systems knowledge can understand. 
    The information you are provided contain the {domain} preferences of the group. Every candidate item for recommendation has a rating from each user listed in the order by user (first rating from user1, second from user2 etc). 
    The rating is a scale from 0 to 100. For the recommendation, you simply mention the {domain}.
    You make a recommendation to the group of users by providing a ranking of 10 {domain} based on the recommendation approach you came up with. 

    Provide your answer as VALID JSON ONLY.
    Do not use markdown or code fences.
    Do not include newlines inside string values.
    Use plain ASCII characters only.

    Format:
        {{
        "recommendation": ["item1","item2","item3","item4","item5","item6","item7","item8","item9","item10"],
        "explanation": Short explanation of how you made the recommendation with no line breaks"
        }}

    """}
        
        ### for strats ###

        restaurants = matrix[0].keys()
        result = {r: [entry[r] for entry in matrix] for r in restaurants}

        rating_rows = []
        for item, ratings in result.items():
            for r in ratings:
                if isinstance(r, list):
                    rating_rows.extend([{"item": item, "rating": v} for v in r])
                else:
                    rating_rows.append({"item": item, "rating": r})

        rating_df = pd.DataFrame(rating_rows)

        strategy_outputs = {}

        strategy_outputs = {s: strategies[s](rating_df) for s in ['ADD','APP','LMS','MPL','MAJ']}

        
        ###

        messages = []
        messages.append(system_message)
        scenario = {
            'role': 'user',
            'content': f"""
            The per-item ratings are presente below:
            ### BEGIN TABLE ###
            {result}
            ### END TABLE ###

            Think about the answer internally, but only output the final JSON object (containing recommendation ranking and explanation). Do not include any additional text or python code. 
            Return STRICT JSON. Do not use markdown.
            """

        }
        messages.append(scenario)
        chat_instance = messages
        if 'gpt' in llm_name.lower(): #gpt oss
            resp = client_ol.chat(llm_name, messages=messages,options={"temperature":0.6, 'top_p':0.9})
            out = resp['message']['content']
        elif 'stral' in llm_name.lower(): #mistral, ministral
            resp = client_ol.chat(llm_name, messages=messages,options={"temperature":0.6, 'top_p':0.9})
            out = resp['message']['content']    
        else:
            resp = client.chat.completions.create(
                                        messages = chat_instance,
                                        model=llm_name,
                                        stream=False,
                                        max_completion_tokens=1000,
                                        temperature=0.6,
                                        top_p=0.9
                                    )

            out = resp.choices[0].message.content
        if "<think>" in out or "</think>" in out:
            out = re.sub(r'^.*?</think>', '', out, flags=re.DOTALL).strip()
        out = parse_llm_json(out)
        #print(out)
        recommendation = out['recommendation']
        explanation = out['explanation']

        row = {
        'configuration': configuration,
        'domain': domain,
        'llm': llm_name,
        'recommendation': recommendation,
        'explanation': explanation
            }

        for strategy in ['ADD', 'APP', 'LMS', 'MPL', 'MAJ']:
            row[strategy] = strategy_outputs[strategy]

        df_results = pd.concat([df_results, pd.DataFrame([{
        'configuration': configuration,
        'group_size':group_size,
        'num_items':num_items,
        'domain': domain,
        'llm': llm_name,
        'recommendation': recommendation,
        'explanation': explanation,
        'ADD': row['ADD'],
        'APP': row['APP'],
        'LMS': row['LMS'],
        'MPL': row['MPL'],
        'MAJ': row['MAJ'],
        'group': str(result),
        }])], ignore_index=True)
        df_results.to_csv('results.csv', index=False)
        time.sleep(2)
    except RateLimitError as e:
        errorcount +=1
        msg = str(e)
        ers.append(msg)
        if errorcount > 5:
            print('Time to stop. Error count above 5')
            print(ers)
            break
        
        if "high traffic" in msg.lower() or "queue_exceeded" in msg.lower():
            print(f"High traffic detected.")
            time.sleep(20)
            continue


In [ ]:
## OLD: now BORDA and AWM included in strats.py!
## RUN TO ADD BORDA + AWM REC LISTS

import ast

file_path = 'results.csv'
df_results = pd.read_csv(file_path)
def select_top_n(data, value_col=None, item_col='item', n=1, random_state=None, mode="random"):
    rng = np.random.default_rng(random_state)
    
    if isinstance(data, list):
        if len(data) <= n:
            return data.copy()
        if mode == "consensus":
            return data.copy() 
        else:  # random
            return rng.choice(data, size=n, replace=False).tolist()

    sorted_df = data.sort_values(value_col, ascending=False).reset_index(drop=True)

    if mode == "consensus":
        top_value = sorted_df[value_col].iloc[0]
        tied_items = sorted_df.loc[sorted_df[value_col] == top_value, item_col].tolist()
        return tied_items

    results = []
    i = 0
    while len(results) < n and i < len(sorted_df):
        tied_items = sorted_df.loc[sorted_df[value_col] == sorted_df.loc[i, value_col], item_col].tolist()
        remaining_slots = n - len(results)
        if len(tied_items) <= remaining_slots:
            results.extend(tied_items)
        else:
            chosen = rng.choice(tied_items, size=remaining_slots, replace=False).tolist()
            results.extend(chosen)
        i += len(tied_items)
    return results
def BORDA(ratings_dict, n=1,mode="random"):
    df = pd.DataFrame(ratings_dict)  
    borda_scores = pd.Series(0, index=df.columns)

    for _, row in df.iterrows():  
        ranked_items = row.rank(method='min', ascending=False)  
        points = (len(row) - ranked_items)
        borda_scores += points

    scores_df = borda_scores.reset_index()
    scores_df.columns = ['item', 'borda_score']
    return select_top_n(scores_df, 'borda_score', n=n, random_state=123,mode=mode)

def AWM(df, threshold=4, n=1, mode="random"):
    avg_ratings = df.groupby('item')['rating'].mean().reset_index(name='avg_rating')
    valid_items = df.groupby('item')['rating'].min().reset_index(name='min_rating')
    valid_items = valid_items[valid_items['min_rating'] >= threshold]
    filtered = avg_ratings.merge(valid_items[['item']], on='item', how='inner')
    if filtered.empty:
        return []

    return select_top_n(filtered, 'avg_rating', n=n, random_state=123, mode=mode)
def group_dict_to_df(group_dict):
    return pd.DataFrame(
        [(item, rating) for item, ratings in group_dict.items() for rating in ratings],
        columns=['item', 'rating']
    )
def apply_borda(row, n=10, mode="random"):
    ratings_dict = row['group_dict']
    return BORDA(ratings_dict, n=n, mode=mode)
def apply_awm(row, threshold=4, n=10, mode="random"):
    group_df = group_dict_to_df(row['group_dict'])
    return AWM(group_df, threshold=threshold, n=n, mode=mode)



df_results['group_dict'] = df_results['group'].apply(ast.literal_eval)
df_results['BORDA'] = df_results.apply(apply_borda, axis=1)
df_results['AWM'] = df_results.apply(apply_awm, axis=1)


In [14]:
import re
import pandas as pd
import numpy as np
import spacy
from rapidfuzz import process, fuzz
from nltk.stem import WordNetLemmatizer
nlp = spacy.load("en_core_web_sm")
lemmatizer = WordNetLemmatizer()

categories = {
    'APP': ['threshold', 'above a rating', 'consistently high'],
    'Ave': ['high average', 'average rating', 'average score', 'averaged'],
    'ADD': ['sum', 'add up', 'add'],
    'MPL': ['highest rating', 'overall highest', 'liked the most'],
    'Div': ['diversity', 'wide range', 'high variance', 'spread', 'diverse', 'standard deviation'],
    'Sim': ['similar user', 'similar rating', 'similar group', 'similar taste', 'similar item', 'similar'],
    'Pop': ['liked by multiple users', 'rated highly by multiple', 'many users have given high ratings to',
                              'generally high rating', 'highly rated', 'consistently high','high ratings across','popular'],
    'LMS': ['highest of lowest values', 'high minimum'],
    'AWM': ['no very low', 'very low ratings', 'low rating unfairly', 'lowest rating', 'very low', 'extreme low'],
    'FAI': ['fairness', 'fair'],
    #'WEI': ['weight'],
    'BAL': ['balanced appeal', 'consistency', 'moderate', 'balanced approach', 'balanced two']

}


def preprocess_text(text):
    #lemmatization to handle text
    doc = nlp(text.lower())  
    return " ".join([lemmatizer.lemmatize(token.text) for token in doc])

def extract_numbers(text):
    # handle above 60, above 50 etc
    return [int(num) for num in re.findall(r'\b\d+\b', text)]

def is_negated(text, keyword):
    #check if keywords are negated
    pattern = r'\b(not|never|without|did\snot)\s+' + re.escape(keyword) + r'\b'
    return bool(re.search(pattern, text))

def fuzzy_match(text, category_keywords, threshold=85):
    # fuzzy match similarity
    for keyword in category_keywords:
        if fuzz.partial_ratio(text, keyword) >= threshold:
            return True
    return False

#main func
def categorize_explanation(text):
    if pd.isna(text):  
        return "N/A"
    
    text = preprocess_text(text)  # preproc
    
    matched_categories = set()
    
    for category, keywords in categories.items():
        for keyword in keywords:
            keyword = preprocess_text(keyword) 
            
           
            if re.search(r'\b\d+\b', keyword):  
                text_nums = extract_numbers(text)
                keyword_nums = extract_numbers(keyword)
                if text_nums and keyword_nums:
                    if all(t >= k for t, k in zip(text_nums, keyword_nums)):  
                        matched_categories.add(category)
                        break
            
            
            if fuzzy_match(text, [keyword]):
                if is_negated(text, keyword):
                    break  # Skip if negated
                matched_categories.add(category)
                break  

    # exclusion rules
    if 'Pop' in matched_categories and ('Ave' in matched_categories or 'App' in matched_categories):
        matched_categories.remove('Pop')


    if 'ADD' in matched_categories and 'Ave' in matched_categories:
        matched_categories.remove('ADD')
    if 'AWM' in matched_categories and 'Ave' in matched_categories:
        matched_categories.remove('Ave')
        

    return ', '.join(matched_categories) if matched_categories else "Other"





e_m = df_results

e_m['labels'] = e_m['explanation'].apply(categorize_explanation)
#e_m

In [ ]:
df_results.to_csv('results.csv', index=False)
